# Simulated PAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 1  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_phase = np.angle(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i =  np.random.uniform(0.5, 2)  # random freq
    phi_i = np.random.uniform(-2 * np.pi, 2 * np.pi)
    a_i= np.random.uniform(1,5)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)* a_i
amp_mod =.25 + .75*(amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]
# Each component has a random frequency and random phase.
# then normalize the total signal to be in range [0, 1].
# amp_mod will control how much the LF envelope drives the HF envelope.

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2*np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
# Implements A_H(t) = (1 - mod) * hf_amp_ind + mod * f(φ_L(t))
sigma = np.pi / 2
f_phi_l = np.exp(-((lf_phase - np.pi) ** 2) / (2 * sigma ** 2))  # f(φ_L(t))
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
# plt.plot(times, lf_signal, label='lf_signal', linestyle ='--')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
# plt.plot(times, lf_phase, label='lf_phase(t)', linestyle=':')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.plot(times, f_phi_l, label='f(ϕ_L(t))', linestyle=':')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()


# Add Word Onset Modulation
mod(f) = (WORD ONSETS) * k(t)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 1  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_phase = np.angle(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate amp_mod(t) from word onsets and kernel ---
word_onsets = np.zeros(n_samples)
word_locs = np.random.randint(100, 900, size=5)  # random word onset indices
word_onsets[word_locs] = 1

# Gaussian kernel: spreads effect of each word over ~100 ms
kernel_width = int(0.1 * sfreq)  # 100 ms
kernel = np.exp(-np.linspace(-1, 1, 2 * kernel_width)**2 / 0.05)
kernel = kernel / kernel.max()  # normalize to [0, 1]

# Convolve to get slow modulation function
amp_mod = np.convolve(word_onsets, kernel, mode='same')
amp_mod = 0.25 + 0.75 * (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2*np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
# Implements A_H(t) = (1 - mod) * hf_amp_ind + mod * f(φ_L(t))
sigma = np.pi / 2
f_phi_l = np.exp(-((lf_phase - np.pi) ** 2) / (2 * sigma ** 2))  # f(φ_L(t))
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
# plt.plot(times, lf_signal, label='lf_signal', linestyle ='--')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
# plt.plot(times, lf_phase, label='lf_phase(t)', linestyle=':')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.plot(times, f_phi_l, label='f(ϕ_L(t))', linestyle=':')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 1  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_phase = np.angle(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i =  np.random.uniform(0.5, 2)  # random freq
    phi_i = np.random.uniform(-2 * np.pi, 2 * np.pi)
    a_i= np.random.uniform(1,5)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)* a_i
amp_mod =.25 + .75*(amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]
# generate a slow-varying envelope amp_mod(t) by summing 5 random low-frequency sinusoids (< 1 Hz).
# Each component has a random frequency and random phase.
# then normalize the total signal to be in range [0, 1].
# amp_mod will control how much the LF envelope drives the HF envelope.

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2*np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
# === Theoretical model: A_H(t) = c * [(1 - mod(f)) + mod(f) * f(φ_L(t))] ===
sigma = np.pi / 2
f_phi_l = np.exp(-((lf_phase - np.pi) ** 2) / (2 * sigma ** 2))  # f(φ_L(t))

word_onsets = np.zeros_like(times)
word_onsets[np.random.randint(100, 900, 5)] = 1  # 5 spikes
kernel = np.exp(-np.linspace(-1, 1, 200)**2 / 0.02)
mod_f = np.convolve(word_onsets, kernel, mode='same')
mod_f = mod_f / mod_f.max()  # normalize

# Use this instead of amp_mod:
hf_envelope_word = (1 - mod_f) * hf_amp_ind + mod_f * f_phi_l


# hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l  # matches formula
# blend hf_amp_ind and lf_amp using amp_mod as a mixing coefficient
# At time t, if amp_mod[t] is close to 0, then hf_envelope[t] ≈ hf_amp_ind[t].
# If amp_mod[t] is close to 1, then hf_envelope[t] ≈ lf_amp[t].
# This simulates intermittent PAC, rather than constant coupling — realistic EEG-inspired design.

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
plt.plot(times, lf_signal, label='lf_signal', linestyle ='--')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
plt.plot(times, lf_phase, label='lf_phase(t)', linestyle=':')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()




# Create 3 subplots showing different mechanisms of PAC:

# Constant HF background
# HF only on at LF onsets
# HF amplitude modulated by LF phase

fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

# (1) baseline hf
axs[0].plot(times, hf_carrier, label='HF (baseline)', color='gray')
axs[0].legend()

# (2) HF only on at phi_L(t) = π
hf_onset_mask = (np.abs(lf_phase - np.pi) < 0.3).astype(float)
hf_onset_signal = hf_carrier * hf_onset_mask
axs[1].plot(times, hf_onset_signal, label='HF only at LF phase ~ π', color='green')
axs[1].legend()

# (3) HF modulated by f(φ_L)
hf_modulated_signal = hf_carrier * f_phi_l
axs[2].plot(times, hf_modulated_signal, label='HF modulated by LF phase', color='magenta')
axs[2].legend()

for ax in axs:
    ax.set_ylim(-1.5, 1.5)
    ax.axvline(0.25, linestyle='--', color='k')
    ax.axvline(0.5, linestyle='--', color='k')
    ax.axvline(0.75, linestyle='--', color='k')

plt.suptitle('PAC Mechanisms Visualization')
plt.xlabel('Time (s)')
plt.tight_layout()
plt.show()








# Simulated AAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 10  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier + fluctuations
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_amp = np.abs(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i = np.random.uniform(0.05, 0.5)  # random freq <= 0.5 Hz
    phi_i = np.random.uniform(0, 2 * np.pi)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)
amp_mod = (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]
# generate a slow-varying envelope amp_mod(t) by summing 5 random low-frequency sinusoids (< 1 Hz).
# Each component has a random frequency and random phase.
# then normalize the total signal to be in range [0, 1].
# amp_mod will control how much the LF envelope drives the HF envelope.

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(0, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * lf_amp
# blend hf_amp_ind and lf_amp using amp_mod as a mixing coefficient
# At time t, if amp_mod[t] is close to 0, then hf_envelope[t] ≈ hf_amp_ind[t].
# If amp_mod[t] is close to 1, then hf_envelope[t] ≈ lf_amp[t].
# This simulates intermittent PAC, rather than constant coupling — realistic EEG-inspired design.

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='--')
plt.plot(times, lf_amp, label='lf_amp(t)', linestyle='--')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt

# --- Setup ---
sfreq = 1000
duration = 2.5
times = np.arange(0, duration, 1/sfreq)
n_samples = len(times)

# Low-frequency signal
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_phase = np.angle(hilbert(lf_signal))

# High-frequency carrier
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times)

# Amplitude modulation by LF phase
sigma = np.pi / 2
modulator = np.exp(-((lf_phase - np.pi)**2) / (2 * sigma**2))
hf_signal = (1 + 0.5 * modulator) * hf_carrier + 0.3 * np.random.randn(n_samples)

plt.figure(figsize=(6, 2))
plt.plot(times, hf_signal, color='gray')
plt.plot(times, (1 + 0.5 * modulator), color='red', lw=2)
for t in [1.0, 1.5, 2.0]:
    plt.axvline(t, color='black', linestyle='--')
plt.title("Simulated PAC mechanisms")
plt.xlim(1.0, 2.5)
plt.tight_layout()
plt.show()

# Feature 1: Global PAC (no feature mod)
feat1 = np.zeros_like(times)

# Feature 2: Binary onsets
feat2 = np.zeros_like(times)
onsets = [1.1, 1.6, 2.1]
for t in onsets:
    feat2[np.argmin(np.abs(times - t))] = 1

# Feature 3: Value-modulated
feat3 = np.zeros_like(times)
for t in onsets:
    feat3[np.argmin(np.abs(times - t))] = np.random.uniform(0.5, 1.5)

plt.figure(figsize=(6, 3))
plt.plot(times, hf_signal / np.max(hf_signal) + 1.5, label='(1) PAC Signal')
plt.plot(times, feat2 + 1.0, label='(2) Onsets', color='green')
plt.plot(times, feat3 + 0.5, label='(3) Valued', color='magenta')
for t in onsets:
    plt.axvline(t, linestyle='--', color='k')
plt.title("Simulated features")
plt.xlim(1.0, 2.5)
plt.yticks([])
plt.tight_layout()
plt.legend()
plt.show()

# Extract analytic signal of HF signal
hf_env = np.abs(hilbert(hf_signal))
analytic = hf_env * np.exp(1j * lf_phase)

def build_lag_matrix(feature, lags):
    X = []
    for lag in lags:
        shifted = np.roll(feature, lag)
        if lag < 0:
            shifted[lag:] = 0
        elif lag > 0:
            shifted[:lag] = 0
        X.append(shifted)
    return np.array(X).T

def pac_trf(feature, analytic, lags):
    X = build_lag_matrix(feature, lags)
    Y = analytic
    beta_real = np.linalg.lstsq(X, Y.real, rcond=None)[0]
    beta_imag = np.linalg.lstsq(X, Y.imag, rcond=None)[0]
    pac_magnitude = np.sqrt(beta_real**2 + beta_imag**2)
    return pac_magnitude

lags = np.arange(-100, 200)  # -100ms to +200ms

fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i, (label, feat_on, feat_val) in enumerate([
    ("(1) No feature mod", np.zeros_like(times), np.zeros_like(times)),
    ("(2) Onset mod", feat2, np.zeros_like(times)),
    ("(3) Valued mod", feat2, feat3)
]):
    pac_onset = pac_trf(feat_on, analytic, lags)
    pac_valued = pac_trf(feat_val, analytic, lags)

    axs[i].plot(lags / sfreq, pac_onset, label="Onsets", color="green")
    axs[i].plot(lags / sfreq, pac_valued, label="Valued", color="magenta")
    axs[i].set_title(f"{label}")
    axs[i].set_xlabel("Lag (s)")
    axs[i].set_xlim(-0.1, 0.2)
axs[0].set_ylabel("PAC (a.u.)")
axs[0].legend()
plt.tight_layout()
plt.show()
